Step 1: Import Libraries

In [0]:
from pyspark.sql.functions import *

Step 2: Read Bronze Tables

In [0]:
sales_df = spark.table(
    "retail_analytics_catalog.bronze.sales_raw"
)

customer_df = spark.table(
    "retail_analytics_catalog.bronze.customers_raw"
)

product_df = spark.table(
    "retail_analytics_catalog.bronze.products_raw"
)

Step 3: Apply Validations Again

In [0]:
valid_sales = sales_df.filter(
    col("customer_id").isNotNull()
    &
    col("product_id").isNotNull()
    &
    (col("quantity") > 0)
)

Step 4: Validate Customer IDs

In [0]:
valid_sales = (
    valid_sales
    .join(
        customer_df.select("customer_id"),
        "customer_id",
        "inner"
    )
)

Step 5: Validate Product IDs

In [0]:
valid_sales = (
    valid_sales
    .join(
        product_df.select("product_id"),
        "product_id",
        "inner"
    )
)

Step 6: Remove Duplicate Orders

In [0]:
valid_sales = valid_sales.dropDuplicates(
    ["order_id"]
)

Step 7: Join Customer Details

In [0]:
silver_df = (
    valid_sales
    .join(
        customer_df,
        "customer_id",
        "left"
    )
)

Step 8: Join Product Details

In [0]:
silver_df = (
    silver_df
    .join(
        product_df,
        "product_id",
        "left"
    )
)

Step 9: Create Business Column

In [0]:
silver_df = silver_df.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

Step 10: Create Order Month

In [0]:
silver_df = silver_df.withColumn(
    "order_month",
    month(col("order_date"))
)

Step 11: Create Order Year

In [0]:
silver_df = silver_df.withColumn(
    "order_year",
    year(col("order_date"))
)

Step 12: Save Silver Table

In [0]:
silver_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
    "retail_analytics_catalog.silver.sales_clean"
)

Step 13: Verify Silver Data

In [0]:
display(
    spark.table(
        "retail_analytics_catalog.silver.sales_clean"
    )
)

Step 14: Verify Record Counts

In [0]:
source_count = sales_df.count()

silver_count = spark.table(
    "retail_analytics_catalog.silver.sales_clean"
).count()

print("Bronze Records :", source_count)

print("Silver Records :", silver_count)

Step 15: Verify Table Created

In [0]:
%sql
SHOW TABLES IN retail_analytics_catalog.silver;